# Task 3 — Clean-Slate Screen 1

This notebook runs the first clean-slate model screen. It does not continue the old CNN chain. Gender and usage use separate model designs, and only canonical folds 0 and 4 are scored at this stage. The human observability review is deferred and does not block this run.


## 1. Colab and Drive paths

The fixed-feature models run on CPU. A GPU runtime is not required. Large working cache files are built on Colab's local `/content` disk, then only completed caches, models, predictions, and registry rows are copied to Drive.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import zipfile

REPO_URL = 'https://github.com/TrnLin/MLA2.git'
BRANCH = 'task-3-gender-usage-classification'
REPO_DIR = Path('/content/MLA2')
DRIVE_MOUNT = Path('/content/drive')
DRIVE_PROJECT_DIR = DRIVE_MOUNT / 'MyDrive/MLA2'
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / 'task3'
DRIVE_REGISTRY = DRIVE_TASK_DIR / 'results/runs.csv'
LOCAL_REGISTRY = REPO_DIR / 'results/runs.csv'
DATA_ZIP = DRIVE_PROJECT_DIR / 'data/task3-data.zip'
LOCAL_FEATURE_WORK_DIR = Path('/content/task3-clean-slate-feature-work')

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Connect this notebook to a Google Colab runtime first.') from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / '.git').is_dir():
    remote_url = subprocess.check_output(
        ['git', 'remote', 'get-url', 'origin'], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f'{REPO_DIR} belongs to a different repository: {remote_url}')
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError('The Colab clone has local changes; repository update stopped.')
    run_checked(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run_checked(['git', 'switch', BRANCH], cwd=REPO_DIR)
    run_checked(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository.')
else:
    run_checked(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR])

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print(f'Repository ready: {REPO_DIR}')
print(f'Branch: {BRANCH}')
print(f'Commit: {commit}')


In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f'Dataset archive not found: {DATA_ZIP}')

teacher_dir = REPO_DIR / 'data/raw/teacher'
required_files = (
    teacher_dir / 'train/styles_train.csv',
    teacher_dir / 'test/styles_prediction.csv',
)
image_dirs = (teacher_dir / 'train/images_train', teacher_dir / 'test/images_test')
image_suffixes = {'.jpg', '.jpeg'}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe = [name for name in names if Path(name).is_absolute() or '..' in Path(name).parts]
    if unsafe:
        raise RuntimeError('The dataset archive contains an unsafe path.')
    expected_images = sum(
        name.startswith('data/raw/teacher/') and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError('The archive has no teacher images in the expected folder.')
    current_images = sum(
        path.suffix.lower() in image_suffixes
        for image_dir in image_dirs
        for path in image_dir.glob('*')
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        print(f'Extracting {expected_images:,} teacher images...', flush=True)
        archive.extractall(REPO_DIR)
    else:
        print('Teacher data is already extracted; skipping.')

actual_images = sum(
    path.suffix.lower() in image_suffixes
    for image_dir in image_dirs
    for path in image_dir.glob('*')
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f'Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; '
        f'missing files: {missing_files}'
    )
print(f'Teacher data ready: {actual_images:,} images')


## 2. Zero-fit preflight

This check reads the canonical split and class maps. It confirms the two distinct model contracts and the 7 GiB host-memory budget. It does not extract the full cache or fit a model.


In [ ]:
required_modules = ('numpy', 'pandas', 'PIL', 'skimage', 'sklearn')
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(f'Colab is missing required packages: {missing_modules}')

os.chdir(REPO_DIR)
os.environ['FASHION_PROJECT_ROOT'] = str(REPO_DIR)
source_dir = str(REPO_DIR / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
DRIVE_TASK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_REGISTRY.parent.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_clean_slate import (
    check_clean_slate_screen_setup,
    prepare_clean_slate_screen_features,
    run_clean_slate_gender_screen,
    run_clean_slate_usage_screen,
)

preflight = check_clean_slate_screen_setup(root=REPO_DIR, folds=(0, 4))
if not preflight['training_screen_ready'] or preflight['training_blockers']:
    raise RuntimeError(f"Training is blocked: {preflight['training_blockers']}")
if preflight['model_fits'] != 0 or preflight['optimizer_steps'] != 0:
    raise RuntimeError('Preflight unexpectedly fitted a model.')
print('Human review:', preflight['human_observability_review_status'])
print('Gender model:', preflight['gender_model'])
print('Usage model:', preflight['usage_model'])
print('Screen folds:', preflight['folds'])
print('Estimated cache GiB:', round(preflight['estimated_two_view_cache_bytes'] / 1024**3, 3))


## 3. Frozen hypotheses

**Gender.** The old learned models nearly memorised their training rows. A fixed foreground-gradient representation has much lower freedom, while keeping the shape and colour signals supported by the EDA. A calibrated linear SVM may therefore reduce the train–validation gap without losing too much gender macro-F1.

**Usage.** Usage is strongly tied to product type, but the true article-type label is not available at Task 3 inference time. This model first predicts a full article-type probability distribution from the image. It then combines that uncertainty with a smoothed `P(usage | articleType)` table fitted only on the outer-fold training rows. This tests a task-shaped decision rule instead of another flat usage head.

These are independent clean-slate candidates. Neither is a child of E1–E10.


## 4. Build or reuse teacher-only feature caches

This is the longest preparation step. It hashes the teacher development images, then writes two label-blind feature matrices to Drive. A valid completed cache is reused on later runs. This cell does not fit either model.


In [ ]:
prepared_features = prepare_clean_slate_screen_features(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    local_work_dir=LOCAL_FEATURE_WORK_DIR,
)
print('Audit hash:', prepared_features['audit_contract']['audit_contract_hash'])
print('Gender feature cache:', prepared_features['gender'])
print('Usage feature cache:', prepared_features['usage'])


## 5. Train Gender Screen 1

For each outer fold, the small `C` grid and sigmoid calibration reuse the four remaining canonical folds as inner checks. No new split is created. Completed matching folds are reused after a disconnect.


In [ ]:
gender_anchor = (
    DRIVE_TASK_DIR
    / 'experiments/t3_gender_e9_semantic_filter/gender/aggregate/oof_predictions.csv'
)
gender_screen = run_clean_slate_gender_screen(
    prepared_features=prepared_features,
    output_root=DRIVE_TASK_DIR,
    folds=(0, 4),
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    anchor_prediction_path=gender_anchor,
    reuse_completed=True,
)
print('Gender metrics:', gender_screen['metrics_path'])
print('Gender screen gate:', gender_screen['metrics']['screen_gate'])


## 6. Train Usage Screen 1

The article-type classifier and type-to-usage table are refitted independently inside each outer fold. Hyperparameter checks reuse the four remaining canonical folds, so no new split is created. No true validation article type is used to make a usage prediction. Completed matching folds are reused after a disconnect.


In [ ]:
usage_anchor = (
    DRIVE_TASK_DIR
    / 'experiments/t3_usage_e8_translation/usage/aggregate/oof_predictions.csv'
)
usage_screen = run_clean_slate_usage_screen(
    prepared_features=prepared_features,
    output_root=DRIVE_TASK_DIR,
    folds=(0, 4),
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    anchor_prediction_path=usage_anchor,
    reuse_completed=True,
)
print('Usage metrics:', usage_screen['metrics_path'])
print('Usage screen gate:', usage_screen['metrics']['screen_gate'])


## 7. Two-fold screen summary

These values cover folds 0 and 4 only. They must not be compared with a five-fold aggregate. The saved gate uses matched rows from the historical anchor when that Drive artifact is available.


In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'target': result['target'],
        'model_family': result['metrics']['model_family'],
        'folds': result['metrics']['validation_folds'],
        'macro_f1': result['metrics']['macro_f1'],
        'fold_sd': result['metrics']['fold_macro_f1_sample_sd'],
        'screen_gate': result['metrics']['screen_gate']['status'],
    }
    for result in (gender_screen, usage_screen)
])
display(summary)
print('Stop here. Review the saved evidence before advancing either model to five folds.')
